# config

In [5]:
DATASET_PATH = "C:\\Users\\tombe\\Documents\\_MLE\\CV-for-GRIT\\models\\datasets\\Coleoptera"
MODEL_PATH = "C:\\Users\\tombe\\Documents\\_MLE\\CV-for-GRIT\\models\\LoRA\\checkpoints\\lora_coleoptera.pt"

## imports

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms

import json
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from models.LoRA.lora_config import LoRAConfig, TrainingConfig, InsectConfig

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device : {DEVICE}")
print(f"pytorch : {torch.__version__}")

lora_cfg = LoRAConfig()
train_cfg = TrainingConfig()
insect_cfg = InsectConfig()

print("configuration lora :")
print(json.dumps(asdict(lora_cfg), indent=2))
print("\ngroupes d'insectes :")
for grp in insect_cfg.groups:
    print(f"  • {grp}")


device : cpu
pytorch : 2.11.0+cpu
configuration lora :
{
  "rank": 8,
  "alpha": 16.0,
  "dropout": 0.05,
  "target_modules": [
    "Conv2d",
    "Linear"
  ],
  "min_weight_size": 16
}

groupes d'insectes :
  • hymenoptera


## loading

In [ ]:
class LoRAWeightManager:
    """
    sauvegarde, charge et échange les poids lora par groupe d'insectes.
    seuls lora_A / lora_B / lora_down / lora_up sont stockés → stockage minimal.
    """

    LORA_KEYS = ("lora_A", "lora_B", "lora_down.weight", "lora_up.weight")

    def __init__(self, save_dir: str):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self._cache: Dict[str, Dict[str, torch.Tensor]] = {}

    @staticmethod
    def extract_lora_state(model: nn.Module) -> Dict[str, torch.Tensor]:
        """extrait uniquement les paramètres lora du state_dict."""
        return {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
            if any(key in k for key in LoRAWeightManager.LORA_KEYS)
        }

    @staticmethod
    def apply_lora_state(model: nn.Module, lora_state: Dict[str, torch.Tensor]):
        """charge des poids lora dans le modèle sans modifier les poids de base."""
        current_state = model.state_dict()
        current_state.update(lora_state)
        model.load_state_dict(current_state, strict=False)

    def save_group(self, group_name: str, model: nn.Module, metadata: Optional[dict] = None):
        """sauvegarde les poids lora d'un groupe sur disque."""
        lora_state = self.extract_lora_state(model)
        checkpoint = {
            "group": group_name,
            "lora_state": lora_state,
            "metadata": metadata or {},
            "param_count": sum(v.numel() for v in lora_state.values()),
        }
        path = self.save_dir / f"lora_{group_name}.pt"
        torch.save(checkpoint, path)
        self._cache[group_name] = lora_state
        logger.info(f"  → groupe '{group_name}' sauvegardé : {path} "
                    f"({checkpoint['param_count']:,} params lora)")
        return path

    def load_group(self, group_name: str, model: nn.Module) -> dict:
        """charge et applique les poids lora d'un groupe."""
        if group_name in self._cache:
            self.apply_lora_state(model, self._cache[group_name])
            return {"source": "cache"}

        path = self.save_dir / f"lora_{group_name}.pt"
        if not path.exists():
            raise FileNotFoundError(f"pas de checkpoint pour le groupe '{group_name}' : {path}")
        checkpoint = torch.load(path, map_location=DEVICE)
        self.apply_lora_state(model, checkpoint["lora_state"])
        self._cache[group_name] = checkpoint["lora_state"]
        logger.info(f"  ← groupe '{group_name}' chargé depuis {path}")
        return checkpoint.get("metadata", {})

    def summary(self):
        """affiche un tableau des groupes disponibles sur disque."""
        files = list(self.save_dir.glob("lora_*.pt"))
        if not files:
            print("aucun groupe sauvegardé.")
            return
        print(f"{'Groupe':<20} {'Paramètres LoRA':>18} {'Taille (Mo)':>12}")
        print("-" * 55)
        for f in sorted(files):
            ck = torch.load(f, map_location="cpu")
            size_mb = f.stat().st_size / 1e6
            print(f"{ck['group']:<20} {ck['param_count']:>18,} {size_mb:>12.3f}")


lora_manager = LoRAWeightManager(train_cfg.save_dir)
print("✓ LoRAWeightManager instancié")


dict_keys(['group', 'lora_state', 'metadata', 'param_count'])


## evaluation

In [ ]:
# loading image
image_path = DATASET_PATH + "\\images\\test\\EB6.00J.jpg"
image = Image.open(image_path).convert("RGB")

result=model.predict(image)
print(result)
